# Combining Pitchfork, Discogs, and Reviews

Reviews from https://huggingface.co/datasets/mattismegevand/pitchfork

In [1]:
import pandas as pd

In [8]:
p_d = pd.read_csv("pitchfork_discogs.csv")

reviews = pd.read_csv("reviews.csv")

print("Pitchfork_discogs:", p_d.shape)
print("Reviews dataset:", reviews.shape)

reviews.head()

Pitchfork_discogs: (10764, 32)
Reviews dataset: (25709, 11)


,artist,album,year_released,rating,small_text,review,reviewer,genre,label,reviewed,album_art_url
0,Joni Mitchell,Joni Mitchell at Newport,2023,6.0,The singer’s first full concert in 22 years wa...,Joni Mitchell’s unannounced set at the Newport...,Grayson Haver Currin,Rock,Rhino,"July 29, 2023",https://media.pitchfork.com/photos/645cf82e282...
1,The Clientele,I Am Not There Anymore,2023,8.0,Marked by newfound sonic adventurousness and e...,The Clientele were once the discerning indie c...,Marc Hogan,Rock,Merge,"July 28, 2023",https://media.pitchfork.com/photos/645e61f1af0...
2,Nas,Magic 2,2023,6.5,"Nas and Hit-Boy cook up another collaboration,...",Hit-Boy has brought his ear for samples and sy...,Dylan Green,Rap,Mass Appeal,"July 28, 2023",https://media.pitchfork.com/photos/64bfebaaf65...
3,Beverly Glenn-Copeland,The Ones Ahead,2023,8.0,The Canadian composer’s first new material sin...,The world has never been noisier. For people w...,Jesse Dorris,Experimental,Transgressive,"July 28, 2023",https://media.pitchfork.com/photos/646273b3411...
4,Damon Locks,New Future City Radio,2023,6.9,"Cross-cutting sample collages, beat tracks, an...",Since the broadcast spectrum was cordoned off ...,Matthew Blackwell,Jazz / Experimental,International Anthem,"July 28, 2023",https://media.pitchfork.com/photos/646cc208556...


In [9]:
p_d = p_d.drop(p_d.columns[-9:], axis=1)

In [11]:
p_d["artist_key"] = p_d["artist"].str.lower().str.strip()
p_d["album_key"] = p_d["album"].str.lower().str.strip()

reviews["artist_key"] = reviews["artist"].str.lower().str.strip()
reviews["album_key"] = reviews["album"].str.lower().str.strip()

In [12]:
reviews_unique = reviews.drop_duplicates(
    subset=["artist_key", "album_key"]
).copy()

reviews_unique = reviews_unique[["artist_key", "album_key", "review"]]

merged = p_d.merge(
    reviews_unique,
    on=["artist_key", "album_key"],
    how="left",
    validate="one_to_one"
)

merged = merged.drop(
    columns=["artist_key", "album_key"]
)

merged.head()

,artist,album,reviewauthor,score,releaseyear,reviewdate,recordlabel,genre,danceability,energy,...,instrumentalness,liveness,valence,tempo,date,rating_avg,rating_count,have,want,review
0,Studio 1,Studio 1,Andy Battaglia,8.5,2009.0,February 18 2009,Studio,Electronic,0.511917,0.499667,...,0.024493,0.165367,0.555083,101.395167,2009-02-18,4.42,101.0,483.0,275.0,Wolfgang Voigt is one of techno's most impress...
1,John Fahey,The Great Santa Barbara Oil Slick,Mark Richardson,8.2,2005.0,February 13 2005,Water,Folk/Country,0.369765,0.325412,...,0.559133,0.527782,0.179465,107.622647,2005-02-13,4.52,33.0,400.0,269.0,John Fahey is a tough nut to crack. What was h...
2,Reigning Sound,Too Much Guitar,Stephen M. Deusner,8.3,2004.0,August 19 2004,In the Red,Electronic,0.253943,0.912857,...,0.751214,0.199071,0.552786,133.895500,2004-08-19,4.12,33.0,474.0,94.0,Despite its somewhat questionable reputation a...
3,Mac Miller,Swimming,Evan Rytlewski,7.5,2018.0,August 3 2018,Warner Bros.,Rap,0.624846,0.438154,...,0.012819,0.121131,0.281138,122.121308,2018-08-03,4.76,98.0,1534.0,1664.0,When Ariana Grande left Mac Miller this spring...
4,The Essex Green,Essex Green EP,Matt LeMay,8.1,2000.0,May 31 2000,Parasol,Rock,0.729000,0.932000,...,0.197000,0.139000,0.283000,127.988000,2000-05-31,4.00,12.0,181.0,165.0,There's a wind that blows as winter drifts int...


In [13]:
total = len(merged)
matched = merged["review"].notna().sum()

print(f"Total albums: {total}")
print(f"Reviews matched: {matched}")
print(f"Unmatched albums: {total - matched}")
print(f"Match percentage: {matched / total * 100:.1f}%")

Total albums: 10764
Reviews matched: 10524
Unmatched albums: 240
Match percentage: 97.8%


In [14]:
merged.to_csv(
    "pitchfork_discogs_reviews.csv",
    index=False
)